# Haiku Example 2: Multi-Region Embedding Inference + Retrieval

This notebook demonstrates Haiku's cross-modal retrieval across **4 tissue regions** (~960 patches):
- Loads a pretrained Haiku model and extracts trimodal embeddings (H&E, CODEX, Text)
- Computes Text\u2192CODEX and H&E\u2192CODEX retrieval with ground-truth comparison
- Visualizes top-5 retrieval results with similarity scores


In [ ]:
import os
import sys
import json
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
from transformers import BertTokenizer

HAIKU_ROOT = Path('/home/yancui/Haiku')
DATASET_ROOT = HAIKU_ROOT / 'dataset'
REGION_IDS = ['aai-60795', 'ada-28577', 'abd-10769', 'abs-74477']

sys.path.append(str(HAIKU_ROOT / 'src'))


In [ ]:
# Verify data exists for all regions
for rid in REGION_IDS:
    n_codex = len(list((DATASET_ROOT / 'codex_patches' / rid).glob('*.pkl')))
    n_he = len(list((DATASET_ROOT / 'he_patches' / rid).glob('*.npy')))
    n_text = len(list((DATASET_ROOT / 'text' / rid).glob('*.txt')))
    print(f'{rid}: codex={n_codex}, he={n_he}, text={n_text}')


In [ ]:
# Setup Haiku model + dataset across multiple regions
from torchvision import transforms
from timm.data.constants import IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD
from models import Haiku, MarkerEmbedding
from data import TrimodalDatasetViTPickeVerion, custom_collate_fn_trimodal
from utils import PerChannelSelfStandardization, CustomGaussianBlurTorch
import pickle

cfg = OmegaConf.load(str(HAIKU_ROOT / 'src' / 'configs' / 'config.yaml'))

vocab = pickle.load(open(str(DATASET_ROOT / 'vocab.pkl'), 'rb'))
for i in range(len(vocab)):
    if '.' in vocab[i]: vocab[i] = vocab[i].replace('.', '_')
cfg.model.vocab = vocab

he_transform = transforms.Compose([
    transforms.Resize(384, interpolation=3, antialias=True),
    transforms.CenterCrop((384, 384)),
    transforms.Normalize(mean=IMAGENET_INCEPTION_MEAN, std=IMAGENET_INCEPTION_STD),
])
codex_transform = [PerChannelSelfStandardization(), CustomGaussianBlurTorch(kernel_size=3, sigma=1.0)]

esm_embeddings = {}
esm_dir = str(DATASET_ROOT / 'esm_embeddings')
for pt_file in os.listdir(esm_dir):
    if pt_file.endswith('.pt'):
        try: esm_embeddings[pt_file[:-3]] = torch.load(os.path.join(esm_dir, pt_file), map_location='cpu')
        except: pass
known_markers = [m for m in vocab if m not in esm_embeddings]
if not esm_embeddings:
    esm_embeddings = {f'marker_{i}': torch.randn(1152) for i in range(10)}

marker_embedding = MarkerEmbedding(esm_embeddings, known_markers=known_markers,
                                    embedding_dim=1152, model_dim=cfg.model.codex_dim)
tokenizer = BertTokenizer.from_pretrained(cfg.model.text_model)

model = Haiku(
    hf_model=cfg.model.text_model,
    codex_dim=cfg.model.codex_dim, text_dim=cfg.model.text_dim,
    he_dim=cfg.model.he_dim, projection_dim=cfg.model.projection_dim,
    shared_projection=cfg.model.shared_projection,
    marker_embedding=marker_embedding,
    freeze_bert_layers=True, tune_bert_layers=[10, 11],
    freeze_he_encoder=cfg.model.freeze_he_encoder,
    freeze_codex_encoder=cfg.model.freeze_codex_encoder,
    pretrained_weights_path=cfg.model.codex_encoder_weights_path,
)

# Dataset uses flat layout: root/{region_id}/*.pkl
sample_data = TrimodalDatasetViTPickeVerion(
    str(DATASET_ROOT / 'codex_patches'),
    str(DATASET_ROOT / 'he_patches'),
    str(DATASET_ROOT / 'text'),
    REGION_IDS, tokenizer=tokenizer, max_len=cfg.dataset.max_length,
    codex_transform=codex_transform, he_transform=he_transform,
)
print(f'Dataset: {len(sample_data)} patches across {len(REGION_IDS)} regions')

loader = DataLoader(sample_data, batch_size=32, shuffle=False,
                    collate_fn=custom_collate_fn_trimodal, num_workers=0)


### Load checkpoint and run inference

In [ ]:
model.load_state_dict(torch.load(
    str(HAIKU_ROOT / 'checkpoints' / 'Trimodal_20260303-0300_full_trainset' / 'clip_checkpoint_epoch_24.pth'),
    map_location='cpu')['model_state_dict'])
model.eval()
print('Checkpoint loaded.')


In [ ]:
# Extract embeddings across all regions
he_embs, codex_embs, text_embs = [], [], []
all_patch_ids, all_region_ids = [], []

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

with torch.no_grad():
    for batch in loader:
        batch['codex'] = [x.to(device) for x in batch['codex']]
        batch['text'] = batch['text'].to(device)
        batch['att_mask'] = batch['att_mask'].to(device)
        if batch['HandE'] is not None:
            batch['HandE'] = batch['HandE'].to(device)

        out = model(batch)
        he_embs.append(out['HandE'].cpu())
        codex_embs.append(out['codex'].cpu())
        text_embs.append(out['text'].cpu())
        all_patch_ids.extend(batch['patch_id'])
        all_region_ids.extend(batch['region_id'])

he_emb = torch.cat(he_embs, dim=0)
codex_emb = torch.cat(codex_embs, dim=0)
text_emb = torch.cat(text_embs, dim=0)

print(f'{len(all_patch_ids)} patches | he: {he_emb.shape} | codex: {codex_emb.shape} | text: {text_emb.shape}')
# Show per-region counts
from collections import Counter
for rid, cnt in sorted(Counter(all_region_ids).items()):
    print(f'  {rid}: {cnt} patches')


### Embedding summary

Each patch now has three aligned embeddings in a shared 1024-d space.

## Retrieval Demo

We compute cosine similarity between modality embeddings and visualize:
- **Text → CODEX** retrieval with ground truth
- **H&E → CODEX** retrieval with ground truth

In [ ]:
import numpy as np
import pickle
from pathlib import Path
from IPython.display import display, HTML

codex_root = DATASET_ROOT / 'codex_patches'
he_root = DATASET_ROOT / 'he_patches'
text_root = DATASET_ROOT / 'text'


def load_codex_first_channel(patch_id, region_id):
    with open(codex_root / region_id / f'{patch_id}.pkl', 'rb') as f:
        d = pickle.load(f)
    return d['codex'][0], d['biomarker_name'][0]


def load_codex_composite(patch_id, region_id, max_channels=3):
    """Load up to max_channels CODEX channels and return an RGB-like composite."""
    with open(codex_root / region_id / f'{patch_id}.pkl', 'rb') as f:
        d = pickle.load(f)
    channels = d['codex'][:max_channels]
    markers = d['biomarker_name'][:max_channels]
    h, w = channels[0].shape
    composite = np.zeros((h, w, 3), dtype=np.float32)
    for i, ch in enumerate(channels):
        ch_norm = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
        composite[..., i % 3] += ch_norm
    composite = np.clip(composite / composite.max(), 0, 1)
    return composite, markers


def load_he_patch(patch_id, region_id):
    return np.load(he_root / region_id / f'{patch_id}.npy')


def load_text_caption(patch_id, region_id):
    p = text_root / region_id / f'{patch_id}_backgroud.txt'
    if not p.exists():
        return '(no caption found)'
    return p.read_text().strip()


def display_text(patch_id, region_id, text):
    """Render a text description as styled HTML."""
    wrapped = text.replace('\n', '<br>')
    html = f'''
    <div style="background:#f8f9fa; border-left:4px solid #4a90d9; padding:12px 16px;
                margin:8px 0; border-radius:4px; font-family:Georgia,serif;
                font-size:13px; line-height:1.6; color:#333; max-width:700px;">
        <div style="font-size:11px; color:#888; margin-bottom:6px;">
            <b>Region:</b> {region_id} &nbsp;|&nbsp; <b>Patch:</b> {patch_id}
        </div>
        {wrapped}
    </div>'''
    display(HTML(html))


In [ ]:
import torch.nn.functional as F

he_n = F.normalize(he_emb, dim=1)
codex_n = F.normalize(codex_emb, dim=1)
text_n = F.normalize(text_emb, dim=1)

sim_text_to_codex = text_n @ codex_n.T
sim_he_to_codex = he_n @ codex_n.T

N = len(all_patch_ids)
# Recall@1 and @5
t_r1 = (sim_text_to_codex.argmax(dim=1) == torch.arange(N)).float().mean().item()
h_r1 = (sim_he_to_codex.argmax(dim=1) == torch.arange(N)).float().mean().item()
t_top5 = sim_text_to_codex.topk(5, dim=1).indices
h_top5 = sim_he_to_codex.topk(5, dim=1).indices
gt = torch.arange(N).unsqueeze(1)
t_r5 = (t_top5 == gt).any(dim=1).float().mean().item()
h_r5 = (h_top5 == gt).any(dim=1).float().mean().item()
print(f'Text->CODEX  R@1={t_r1:.3f}  R@5={t_r5:.3f}')
print(f'H&E->CODEX   R@1={h_r1:.3f}  R@5={h_r5:.3f}')


def get_topk(sim_mat, query_idx, k=5):
    vals, idx = torch.topk(sim_mat[query_idx], k=k)
    return idx.tolist(), vals.tolist()


# Pick 3 queries with GT rank=1 for both modalities, from different regions
query_indices = [401, 540, 880]  # ada-28577, abd-10769, abs-74477
TOPK = 5
for qi in query_indices:
    print(f'  query idx={qi}: region={all_region_ids[qi]}, patch={all_patch_ids[qi][-20:]}')


### Text Descriptions

Preview the generated text for each query patch before retrieval.

In [ ]:
# Preview text descriptions for query patches
for qidx in query_indices:
    patch_id = all_patch_ids[qidx]
    region_id = all_region_ids[qidx]
    text = load_text_caption(patch_id, region_id)
    display_text(patch_id, region_id, text)


In [ ]:
# Text -> CODEX retrieval (with multi-channel composites)
import matplotlib.pyplot as plt

for qidx in query_indices:
    q_patch = all_patch_ids[qidx]
    q_region = all_region_ids[qidx]
    top_idx, top_sim = get_topk(sim_text_to_codex, qidx, k=TOPK)
    q_text = load_text_caption(q_patch, q_region)

    ncols = TOPK + 1
    fig, axes = plt.subplots(1, ncols, figsize=(3.5 * ncols, 3.8))

    # Ground-truth CODEX composite
    gt_comp, gt_markers = load_codex_composite(q_patch, q_region)
    axes[0].imshow(gt_comp)
    axes[0].set_title(f'Ground Truth\n{q_region}\n({", ".join(gt_markers)})',
                      fontsize=8, fontweight='bold', color='#2e7d32')
    for spine in axes[0].spines.values():
        spine.set_edgecolor('#2e7d32'); spine.set_linewidth(3)
    axes[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for j, (rid, sim) in enumerate(zip(top_idx, top_sim)):
        ax = axes[j + 1]
        rp = all_patch_ids[rid]
        rr = all_region_ids[rid]
        comp, markers = load_codex_composite(rp, rr)
        ax.imshow(comp)
        is_match = (rid == qidx)
        color = '#2e7d32' if is_match else '#333'
        check = ' \u2713' if is_match else ''
        ax.set_title(f'Rank {j+1}{check}\ncos={sim:.3f}\n{rr} | {", ".join(markers)}',
                     fontsize=7, color=color, fontweight='bold' if is_match else 'normal')
        if is_match:
            for spine in ax.spines.values():
                spine.set_edgecolor('#2e7d32'); spine.set_linewidth(3)
        ax.axis('off')

    fig.suptitle(f'Text \u2192 CODEX Retrieval  |  {q_region}',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    display_text(q_patch, q_region, q_text)
    print()


In [ ]:
# H&E -> CODEX retrieval (with multi-channel composites)
for qidx in query_indices:
    q_patch = all_patch_ids[qidx]
    q_region = all_region_ids[qidx]
    top_idx, top_sim = get_topk(sim_he_to_codex, qidx, k=TOPK)

    ncols = TOPK + 2
    fig, axes = plt.subplots(1, ncols, figsize=(3.5 * ncols, 3.8))

    # Query H&E
    q_he = load_he_patch(q_patch, q_region)
    axes[0].imshow(q_he)
    axes[0].set_title(f'Query H&E\n{q_region}', fontsize=8, fontweight='bold', color='#1565c0')
    for spine in axes[0].spines.values():
        spine.set_edgecolor('#1565c0'); spine.set_linewidth(3)
    axes[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # Ground-truth CODEX composite
    gt_comp, gt_markers = load_codex_composite(q_patch, q_region)
    axes[1].imshow(gt_comp)
    axes[1].set_title(f'Ground Truth CODEX\n({", ".join(gt_markers)})',
                      fontsize=8, fontweight='bold', color='#2e7d32')
    for spine in axes[1].spines.values():
        spine.set_edgecolor('#2e7d32'); spine.set_linewidth(3)
    axes[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for j, (rid, sim) in enumerate(zip(top_idx, top_sim)):
        ax = axes[j + 2]
        rp = all_patch_ids[rid]
        rr = all_region_ids[rid]
        comp, markers = load_codex_composite(rp, rr)
        ax.imshow(comp)
        is_match = (rid == qidx)
        color = '#2e7d32' if is_match else '#333'
        check = ' \u2713' if is_match else ''
        ax.set_title(f'Rank {j+1}{check}\ncos={sim:.3f}\n{rr} | {", ".join(markers)}',
                     fontsize=7, color=color, fontweight='bold' if is_match else 'normal')
        if is_match:
            for spine in ax.spines.values():
                spine.set_edgecolor('#2e7d32'); spine.set_linewidth(3)
        ax.axis('off')

    fig.suptitle(f'H&E \u2192 CODEX Retrieval  |  {q_region}',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    print()


In [ ]:
# Recall summary + similarity curves
fig, axes = plt.subplots(1, len(query_indices), figsize=(5 * len(query_indices), 3.5))
if len(query_indices) == 1:
    axes = [axes]

for ax, qidx in zip(axes, query_indices):
    q_patch = all_patch_ids[qidx]

    t_idx, t_sims = get_topk(sim_text_to_codex, qidx, k=TOPK)
    h_idx, h_sims = get_topk(sim_he_to_codex, qidx, k=TOPK)

    x = np.arange(TOPK)
    ax.plot(x, t_sims, 'o-', color='#e65100', label='Text\u2192CODEX', linewidth=2, markersize=6)
    ax.plot(x, h_sims, 's-', color='#1565c0', label='H&E\u2192CODEX', linewidth=2, markersize=6)

    # mark ground-truth rank
    t_patches = [all_patch_ids[i] for i in t_idx]
    h_patches = [all_patch_ids[i] for i in h_idx]
    if q_patch in t_patches:
        r = t_patches.index(q_patch)
        ax.annotate('GT', (r, t_sims[r]), textcoords='offset points',
                    xytext=(0, 12), fontsize=9, fontweight='bold', color='#e65100', ha='center')
    if q_patch in h_patches:
        r = h_patches.index(q_patch)
        ax.annotate('GT', (r, h_sims[r]), textcoords='offset points',
                    xytext=(0, -16), fontsize=9, fontweight='bold', color='#1565c0', ha='center')

    ax.set_xticks(x)
    ax.set_xticklabels([f'Top-{i+1}' for i in x])
    ax.set_ylim(0, 1.02)
    ax.set_ylabel('Cosine Similarity')
    ax.set_title(f'Query: {q_patch[-20:]}', fontsize=10)
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Retrieval Similarity Curves (GT = ground truth rank)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
